# CICLe Ablation: classes-8

Model: **Llama 3.2-3B** | shots: **1** | variant: **fixed** | alpha: **0.05**

In [1]:
%%capture
# install dependencies:
!pip install numpy matplotlib seaborn torch pandas scikit-learn crepes transformers sentence-transformers accelerate tqdm datasets

### Data processing

In [2]:
import pandas as pd

train_df = pd.read_csv("/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/fol_reasoning_classes-8_train.csv")
test_df = pd.read_csv("/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/fol_reasoning_classes-8_test.csv")

print(f"Train set size: {len(train_df)} examples")
print(f"Test set size: {len(test_df)} examples")

Train set size: 3000 examples
Test set size: 1500 examples


In [3]:
train_df = train_df[train_df['text'].str.strip().astype(bool)]
test_df = test_df[test_df['text'].str.strip().astype(bool)]

train_df.sample()

,text,label
1513,"If something is sugezoti, then it is gupeloco....",2


In [4]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(
    train_df, stratify=train_df["label"], train_size=min(2000, int(0.8*len(train_df))), random_state=42
)

test_df, _ = train_test_split(
    test_df, stratify=test_df["label"], train_size=min(1000, int(0.8*len(test_df))), random_state=42
)

train_df, dev_df = train_test_split(
    train_df, stratify=train_df["label"], test_size=0.2, shuffle=True, random_state=42
)


In [5]:
train_df['label_name'] = train_df['label'].astype(str)
dev_df['label_name'] = dev_df['label'].astype(str)
test_df['label_name'] = test_df['label'].astype(str)

In [6]:
X_train = train_df['text'].to_numpy()
y_train = train_df['label_name'].to_numpy()

X_dev = dev_df['text'].to_numpy()
y_dev = dev_df['label_name'].to_numpy()

X_test = test_df['text'].to_numpy()
y_test = test_df['label_name'].to_numpy()

classes = set(y_train)
print('Classes: ', len(classes))

Classes:  8


In [7]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# load Contriever model
tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")
model = AutoModel.from_pretrained("facebook/contriever")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# mean pooling function
def mean_pooling(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1)

# define embedding function (batched to avoid OOM):
def phi(texts, batch_size=32):
    if hasattr(texts, "tolist"):
        texts = texts.tolist()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

# test embedding shape
phi(X_train[:2].tolist()).shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2, 768)

In [8]:
import numpy as np
id2label = np.unique(y_train)
label2id = {l: i for i, l in enumerate(id2label)}

In [9]:
from crepes import WrapClassifier
from sklearn.linear_model import LogisticRegression

# create a conformal base classifier based on Logistic Regression:
base_classifier = WrapClassifier(LogisticRegression(max_iter=1000))

In [10]:
X_train_emb = phi(X_train)
base_classifier.fit(X_train_emb, [label2id[y] for y in y_train])
base_classifier.calibrate(phi(X_dev), [label2id[y] for y in y_dev], class_cond=True)

WrapClassifier(learner=LogisticRegression(max_iter=1000), calibrated=True, predictor=ConformalClassifier(fitted=True, mondrian=True))

In [11]:
X_test_emb = phi(X_test)
import gc

# free embedding model from GPU to make room for LLM:
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Contriever freed from GPU memory.")

Contriever freed from GPU memory.


In [12]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import json
import numpy as np

def save_results(y_true, y_pred, classes, filename="results.json", set_sizes=None):
    """
    Save classification metrics (report, accuracy, confusion matrix) to JSON.
    set_sizes, if given, is the per-test-example conformal prediction set size
    (len(ps)), used to analyse the effect of the conformal filter on CICLe.
    """
    report = classification_report(
        y_true, y_pred, labels=classes, zero_division=0, output_dict=True
    )
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    results = {
        "classification_report": report,
        "accuracy": accuracy,
        "confusion_matrix": cm.tolist(),
        "labels": classes
    }

    if set_sizes is not None:
        set_sizes = [int(s) for s in set_sizes]
        results["conformal_set_sizes"] = {
            "values": set_sizes,
            "n_empty": sum(1 for s in set_sizes if s == 0),
            "n_singleton": sum(1 for s in set_sizes if s == 1),
            "n_llm": sum(1 for s in set_sizes if s > 1),
            "mean": float(np.mean(set_sizes)),
        }

    with open(filename, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to: '{filename}'")

### Instantiate Llama 3.2-3B

In [13]:
import transformers
import torch

# create llm pipeline:
llm = transformers.pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto"
)
llm.model.generation_config.max_length = None

# Get special tokens for later:
bos_token_id = llm.tokenizer.convert_tokens_to_ids('<|begin_of_text|>')
eos_token_id = llm.tokenizer.convert_tokens_to_ids('<|eot_id|>')
pad_token_id = llm.tokenizer.convert_tokens_to_ids('<|eot_id|>')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [14]:
def create_prompt(text, examples):
    # helper function replacing quotation marks in the text:
    replace_qm = lambda s: s.replace('"', "'")

    # create context:
    context = f'We classify short passages containing logical facts and rules, ending in a yes/no question, into the number of reasoning steps required to answer that question based on their text. Here are some labelled examples sorted from most probable to least probable:\n'

    for x, y, _ in examples:
        context += f'\n"{replace_qm(x)}" => {y}'

    return {"role": "user", "content": f'{context}\n\nPlease predict the correct class for the following sample. Only provide the class label.\n\n"{replace_qm(text)}" => '}

# print sample prompt:
print(create_prompt(X_test[0], [(X_train[0], y_train[0], 1.0)])["content"])

We classify short passages containing logical facts and rules, ending in a yes/no question, into the number of reasoning steps required to answer that question based on their text. Here are some labelled examples sorted from most probable to least probable:

"It is given that Tobinot is secovo. Every secovo thing is tanebomi. It is given that Vebo is vabafi. If something is dicurifo, then it is pecek. If something is rokudeg, then it is konegaf. It is given that Cecikon is tekut. Every pecek thing is citubagep. Zamim is known to be pogim. Anything that is dato is also kodilodo. Anything that is tanebomi is also dicurifo. Question: based on the above, is Tobinot citubagep?" => 4

Please predict the correct class for the following sample. Only provide the class label.

"All things that are vubi are also kitofa. All things that are lenagu are also gecukutir. If something is mole, then it is lenagu. Kosamav is known to be vebitalam. Bidezav is mole. Kofi is known to be kicifu. If something

In [15]:
ps_test = [id2label[s.astype(bool)] for s in base_classifier.predict_set(X_test_emb, confidence=0.95)]

In [16]:
from sklearn.metrics import pairwise_distances

def get_fixed_k_examples(test_idx, prediction_set, k=2):
    test_emb = X_test_emb[test_idx:test_idx+1]
    mask = np.isin(y_train, list(prediction_set))
    if not mask.any():
        return []
    train_embs_masked = X_train_emb[mask]
    y_train_masked = y_train[mask]
    X_train_masked = [x for x, m in zip(X_train, mask) if m]
    similarity = (1. - pairwise_distances(test_emb, train_embs_masked, metric='cosine'))[0]
    top_k_idx = np.argsort(similarity)[::-1][:k]
    return [(X_train_masked[i], y_train_masked[i], similarity[i]) for i in top_k_idx]

In [17]:
from tqdm import tqdm

predictions, set_sizes = [], []
prompt_lengths, input_lengths, shots_lengths, num_shots = [], [], [], []

for i, (x, ps) in tqdm(enumerate(zip(X_test, ps_test)), total=len(X_test)):
    set_sizes.append(len(ps))
    if len(ps) > 1:
        examples = get_fixed_k_examples(i, ps, k=1)
        prompt = create_prompt(x, examples)
        prompt_len = len(llm.tokenizer(prompt['content'])['input_ids'])
        input_len = len(llm.tokenizer(x)['input_ids'])
        shots_text = '\n'.join([e[0] for e in examples])
        shots_len = len(llm.tokenizer(shots_text)['input_ids'])
        prompt_lengths.append(prompt_len); input_lengths.append(input_len)
        shots_lengths.append(shots_len); num_shots.append(len(examples))
        out = llm([prompt], bos_token_id=bos_token_id, eos_token_id=eos_token_id,
                   pad_token_id=pad_token_id, max_new_tokens=5, do_sample=False)
        predictions.append(out[0]['generated_text'][-1]['content'].strip())
    elif len(ps) == 1:
        predictions.append(ps[0])
    else:
        predictions.append(id2label[base_classifier.predict(X_test_emb[i:i+1])[0]])

  0%|          | 0/1000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'bos_token_id', 'eos_token_id', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  0%|          | 1/1000 [00:00<06:09,  2.70it/s]

...

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


100%|█████████▉| 999/1000 [01:20<00:00, 14.38it/s]

Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


100%|██████████| 1000/1000 [01:20<00:00, 12.41it/s]

In [18]:
save_results(y_test, predictions, list(classes), filename="/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/classes-8/results/predictions/fol-ablation-classes-8-llama-3.2-3b-cicle-contriever-lr-1-shots-fixed-0.05-α.json", set_sizes=set_sizes)

Results saved to: '/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/classes-8/results/predictions/fol-ablation-classes-8-llama-3.2-3b-cicle-contriever-lr-1-shots-fixed-0.05-α.json'


In [19]:
import json
import numpy as np

def save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="prompt_lengths.json"):
    """
    Save prompt, input, shots section lengths, and number of shots to a JSON file,
    including summary statistics.
    """
    stats = {
        "prompt_lengths": {
            "mean": float(np.mean(prompt_lengths)),
            "min": int(np.min(prompt_lengths)),
            "max": int(np.max(prompt_lengths)),
            "std": float(np.std(prompt_lengths)),
            "values": prompt_lengths
        },
        "input_lengths": {
            "mean": float(np.mean(input_lengths)),
            "min": int(np.min(input_lengths)),
            "max": int(np.max(input_lengths)),
            "std": float(np.std(input_lengths)),
            "values": input_lengths
        },
        "shots_lengths": {
            "mean": float(np.mean(shots_lengths)),
            "min": int(np.min(shots_lengths)),
            "max": int(np.max(shots_lengths)),
            "std": float(np.std(shots_lengths)),
            "values": shots_lengths
        },
        "num_shots": {
            "mean": float(np.mean(num_shots)),
            "min": int(np.min(num_shots)),
            "max": int(np.max(num_shots)),
            "std": float(np.std(num_shots)),
            "values": num_shots
        }
    }

    with open(filename, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"Prompt length statistics saved to: '{filename}'")


In [20]:
save_prompt_lengths(prompt_lengths, input_lengths, shots_lengths, num_shots, filename="/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/classes-8/results/lengths/fol-ablation-classes-8-llama-3.2-3b-cicle-contriever-lr-1-shots-fixed-0.05-α.json")

Prompt length statistics saved to: '/home/v25/ippa6201/cicle-evaluation/fol-reasoning/ablation/classes-8/results/lengths/fol-ablation-classes-8-llama-3.2-3b-cicle-contriever-lr-1-shots-fixed-0.05-α.json'
